# DanceMatch3 — kostenloser Trainingslauf in Google Colab

Dieses Notebook umgeht GitHub Actions. Es verwendet die bestehende Baseline aus dem Branch `feature/dance-style-training` und läuft im kostenlosen Colab-Runtime, soweit dein Google-Konto dafür Zugriff hat. **Kein GitHub-Actions-Lauf und kein GitHub-Billing nötig.**

Hinweis: Das ist ein erster Forschungs-Baseline-Test, noch kein produktionsreifes Modell für alle neun Tanzstile. Bachata und Discofox brauchen zusätzliche, sauber gelabelte Trainingsdaten.

In [ ]:
# Systembibliotheken und Python-Abhängigkeiten installieren
!apt-get -qq update
!apt-get -qq install -y ffmpeg libsndfile1
!pip -q install 'datasets>=3.0,<5' 'librosa>=0.10.2,<1' 'numpy>=1.26,<2.3' 'scikit-learn>=1.4,<2' 'joblib>=1.3,<2'
print('Setup abgeschlossen')

In [ ]:
# Aktuellen Trainingscode aus deinem GitHub-Branch holen
import os, pathlib
REPO = '/content/DanceMatch3'
if os.path.exists(REPO):
    !git -C /content/DanceMatch3 fetch origin feature/dance-style-training
    !git -C /content/DanceMatch3 checkout feature/dance-style-training
    !git -C /content/DanceMatch3 pull --ff-only origin feature/dance-style-training
else:
    !git clone --depth 1 --branch feature/dance-style-training https://github.com/zwirbel/DanceMatch3.git /content/DanceMatch3\nassert pathlib.Path(REPO, 'training', 'train_baseline.py').exists(), 'Trainingsskript wurde nicht gefunden.'
print('Code bereit:', REPO)

## Trainingsumfang auswählen

`MAX_ITEMS = 0` eignet sich für einen ersten kurzen Funktionstest. Für einen vollständigen Lauf setze den Wert auf `0`. Ein vollständiger Lauf kann je nach Datensatz und Colab-Runtime deutlich länger dauern.

In [ ]:
MAX_ITEMS = 0  # 0 = vollständiger Lauf; positive Zahl = kurzer Funktionstest
DATASET = 'Yi3852/MuCUE'
SPLIT = 'ballroom_genres'
OUT_DIR = f'{REPO}/training/artifacts'
os.makedirs(OUT_DIR, exist_ok=True)
print('MAX_ITEMS =', MAX_ITEMS)

In [ ]:
# Baseline trainieren und Kennzahlen berechnen
%cd /content/DanceMatch3/training
!HF_HUB_DISABLE_TELEMETRY=1 python train_baseline.py --dataset "$DATASET" --split "$SPLIT" --max-items "$MAX_ITEMS" --out artifacts

In [ ]:
# Ergebnisdateien anzeigen und als ZIP herunterladen
from pathlib import Path
import zipfile
artifacts = Path('/content/DanceMatch3/training/artifacts')
files = [p for p in artifacts.rglob('*') if p.is_file()]
for p in files:
    print(p, f'({p.stat().st_size:,} bytes)')
if files:
    zip_path = '/content/dancematch-training-results.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
        for p in files:
            z.write(p, p.relative_to(artifacts))
    from google.colab import files
    files.download(zip_path)
else:
    print('Keine Artefakte gefunden. Bitte Fehlermeldung im Trainingsschritt prüfen.')